# 15.2 Matrix factorization

A sparse rating matrix becomes two small tables of hidden user and item traits whose dot product fills the gaps.

Matrix factorization compresses neighbor evidence into learned coordinates. Gradient descent moves factors after each observed rating error, while regularization prevents the low-rank model from memorizing every visible cell. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1500
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)


def ndcg_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    gains = (2.0 ** relevance[order] - 1.0) / np.log2(np.arange(2, len(order) + 2))
    ideal = np.argsort(-relevance, kind="mergesort")[:k]
    ideal_gains = (2.0 ** relevance[ideal] - 1.0) / np.log2(np.arange(2, len(ideal) + 2))
    denom = np.sum(ideal_gains)
    if denom == 0.0:
        return 0.0
    return float(np.sum(gains) / denom)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def recall_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float) > 0.0
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    denom = np.sum(relevance)
    if denom == 0:
        return 0.0
    return float(np.sum(relevance[order]) / denom)


def rating_ladder():
    d1 = np.array([
        [5.0, 4.0, 0.0, 1.0],
        [4.0, 5.0, 0.0, 1.0],
        [1.0, 1.0, 5.0, 4.0],
        [0.0, 1.0, 4.0, 5.0],
    ])
    d2 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0],
        [4.0, 5.0, 0.0, 1.0, 2.0],
        [1.0, 1.0, 5.0, 4.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0],
        [5.0, 0.0, 1.0, 0.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0],
    ])
    d3 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0, 2.0],
        [4.0, 4.0, 0.0, 0.0, 2.0, 0.0],
        [1.0, 1.0, 5.0, 4.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 3.0, 0.0, 3.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 1.0],
    ])
    d4 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0],
        [4.0, 5.0, 0.0, 2.0, 0.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [1.0, 0.0, 4.0, 5.0, 0.0, 5.0, 0.0, 2.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0],
        [0.0, 3.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 1.0, 4.0, 0.0],
        [0.0, 4.0, 0.0, 5.0, 0.0, 5.0, 0.0, 3.0],
        [4.0, 0.0, 2.0, 0.0, 5.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 4.0, 5.0, 0.0, 4.0, 0.0, 0.0],
    ])
    d5 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [4.0, 5.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 0.0, 5.0, 0.0, 0.0, 0.0, 2.0],
        [0.0, 0.0, 5.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 1.0, 0.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 5.0, 0.0, 3.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 4.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 5.0],
    ])
    return [
        {"name": "D1 tiny rating matrix", "ratings": d1, "holdout": [(0, 2, 5.0)]},
        {"name": "D2 synthetic preferences", "ratings": d2, "holdout": [(0, 2, 5.0), (2, 4, 2.0), (5, 4, 3.0)]},
        {"name": "D3 noise ties sparsity", "ratings": d3, "holdout": [(0, 2, 5.0), (1, 3, 2.0), (6, 1, 3.0), (7, 4, 2.0)]},
        {"name": "D4 inline MovieLens-like sample", "ratings": d4, "holdout": [(0, 2, 4.0), (1, 5, 3.0), (4, 3, 2.0), (6, 7, 2.0)]},
        {"name": "D5 long-tail cold-start", "ratings": d5, "holdout": [(0, 2, 4.0), (2, 9, 2.0), (5, 8, 4.0), (10, 0, 3.0), (11, 1, 3.0)]},
    ]


def slate_ladder():
    return [
        {"name": "D1 3-item slate", "features": np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.744, 0.643, 0.819]), "collab": np.array([0.30, 0.90, 0.70]), "relevance": np.array([2.0, 3.0, 1.0])},
        {"name": "D2 synthetic preferences", "features": np.array([[1.0, 0.1, 0.0], [0.2, 0.9, 0.8], [0.8, 0.7, 0.1], [0.1, 0.2, 1.0]]), "content": np.array([0.70, 0.62, 0.88, 0.55]), "collab": np.array([0.40, 0.85, 0.65, 0.30]), "relevance": np.array([1.0, 3.0, 2.0, 0.0])},
        {"name": "D3 noise ties sparsity", "features": np.array([[1.0, 0.0, 0.2], [0.9, 0.2, 0.0], [0.1, 1.0, 0.9], [0.2, 0.8, 0.8], [0.0, 0.1, 1.0]]), "content": np.array([0.78, 0.78, 0.59, 0.60, 0.51]), "collab": np.array([0.20, 0.50, 0.88, 0.40, 0.10]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 0.0])},
        {"name": "D4 inline MovieLens-like ratings", "features": np.array([[1.0, 0.3, 0.0], [0.9, 0.4, 0.1], [0.2, 0.9, 0.6], [0.1, 0.6, 1.0], [0.7, 0.2, 0.8], [0.0, 0.1, 0.9]]), "content": np.array([0.82, 0.80, 0.66, 0.59, 0.75, 0.50]), "collab": np.array([0.55, 0.61, 0.92, 0.72, 0.35, 0.25]), "relevance": np.array([2.0, 2.0, 3.0, 1.0, 2.0, 0.0])},
        {"name": "D5 long-tail cold-start", "features": np.array([[1.0, 0.2, 0.0], [0.8, 0.1, 0.1], [0.1, 0.9, 0.8], [0.0, 0.7, 1.0], [0.9, 0.9, 0.0], [0.1, 0.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.84, 0.78, 0.61, 0.58, 0.90, 0.52, 0.82]), "collab": np.array([0.65, 0.20, 0.91, 0.55, np.nan, 0.05, np.nan]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 3.0, 0.0, 2.0])},
    ]


def print_ladder_preview(rungs):
    for rung in rungs:
        if "ratings" in rung:
            matrix = rung["ratings"]
            observed = int(np.sum(matrix > 0.0))
            total = int(matrix.size)
            print(rung["name"], matrix.shape, "observed", observed, "of", total)
            print(matrix[:3, :5])
        else:
            relevance = rung["relevance"]
            print(rung["name"], "items", len(relevance), "positives", int(np.sum(relevance > 0.0)))
            print("relevance", relevance[:5])


## The concept, built once: observed-entry low-rank factors

With user factors $P$ and item factors $Q$, the prediction is $$\hat R=PQ^	op,\quad \hat r_{ui}=p_u^	op q_i.$$ The loss is computed only over observed entries $\Omega$, with $L_2$ regularization on both factor tables.

In [ ]:

def fit_mf_sgd(R, rank=2, steps=250, eta=0.03, reg=0.05, seed=0, missing_as_zero=False):
    R = np.asarray(R, dtype=float)
    local_rng = np.random.default_rng(seed)
    P = 0.1 * local_rng.normal(size=(R.shape[0], rank))
    Q = 0.1 * local_rng.normal(size=(R.shape[1], rank))
    if missing_as_zero:
        observations = [(u, i, R[u, i]) for u in range(R.shape[0]) for i in range(R.shape[1])]
    else:
        observations = [(u, i, R[u, i]) for u in range(R.shape[0]) for i in range(R.shape[1]) if R[u, i] > 0.0]
    for step in range(steps):
        order = local_rng.permutation(len(observations))
        for idx in order:
            u, i, rating = observations[idx]
            p_old = P[u].copy()
            q_old = Q[i].copy()
            pred = float(np.dot(p_old, q_old))
            err = rating - pred
            P[u] = p_old + eta * (err * q_old - reg * p_old)
            Q[i] = q_old + eta * (err * p_old - reg * q_old)
    return P, Q


def mf_predict(P, Q, user, item):
    return float(np.dot(P[user], Q[item]))


def one_sgd_step(p, q, rating, eta=0.05, reg=0.1):
    pred = float(np.dot(p, q))
    err = rating - pred
    p_new = p + eta * (err * q - reg * p)
    q_new = q + eta * (err * p - reg * q)
    return pred, err, p_new, q_new


## Check the lesson numbers

The lesson dot products are $4.840$, $0.980$, and $4.830$. One SGD step from $p=(0.8,0.2)$ and $q=(3.0,0.4)$ raises the prediction from $2.480$ to $3.728$ with $p'=(1.174,0.249)$ and $q'=(3.086,0.423)$.

In [ ]:

p0 = np.array([1.2, 0.2])
q0 = np.array([4.0, 0.2])
q2 = np.array([0.2, 3.7])
p2 = np.array([0.1, 1.3])

assert np.round(np.dot(p0, q0), 3) == 4.840
assert np.round(np.dot(p0, q2), 3) == 0.980
assert np.round(np.dot(p2, q2), 3) == 4.830

pred, err, p_new, q_new = one_sgd_step(np.array([0.8, 0.2]), np.array([3.0, 0.4]), 5.0)
new_pred = float(np.dot(p_new, q_new))

assert np.round(pred, 3) == 2.480
assert np.allclose(np.round(p_new, 3), np.array([1.174, 0.249]))
assert np.allclose(np.round(q_new, 3), np.array([3.086, 0.423]))
assert np.round(new_pred, 3) == 3.728

print("lesson dot products", 4.840, 0.980, 4.830)
print("updated prediction", round(new_pred, 3))


## The dataset ladder

The same observed-entry matrix factorizer runs from D1 to D5, including an inline MovieLens-like sample and a long-tail/cold-start matrix with many blanks.

In [ ]:

rungs = rating_ladder()
print_ladder_preview(rungs)


In [ ]:

rows = []
for idx, rung in enumerate(rungs):
    matrix = rung["ratings"].copy()
    true_values = []
    predicted_values = []
    for user, item, true_rating in rung["holdout"]:
        matrix[user, item] = 0.0
        true_values.append(true_rating)
    P, Q = fit_mf_sgd(matrix, rank=2, steps=250, eta=0.03, reg=0.05, seed=SEED + idx)
    for user, item, true_rating in rung["holdout"]:
        predicted_values.append(np.clip(mf_predict(P, Q, user, item), 1.0, 5.0))
    score = rmse(true_values, predicted_values)
    rows.append({"name": rung["name"], "rmse": score, "P": P, "Q": Q, "predictions": predicted_values, "truth": true_values})

for row in rows:
    print(f"{row['name']}: RMSE={row['rmse']:.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
heat = rows[-1]["P"] @ rows[-1]["Q"].T
im = axes[0].imshow(np.clip(heat, 1.0, 5.0), vmin=1.0, vmax=5.0, cmap="viridis")
axes[0].set_title("D5 reconstructed ratings")
axes[0].set_xlabel("item")
axes[0].set_ylabel("user")
fig.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].plot([row["name"].split()[0] for row in rows], [row["rmse"] for row in rows], marker="o")
axes[1].set_title("RMSE vs sparsity rung")
axes[1].set_ylabel("RMSE")
fig.tight_layout()


## Pitfall on D5: blanks as negatives and rank overfit

The wrong model trains on every missing cell as a zero. The fixed model trains on $\Omega$ only and uses a small regularized rank.

In [ ]:

d5 = rating_ladder()[-1]
R5 = d5["ratings"].copy()
for user, item, true_rating in d5["holdout"]:
    R5[user, item] = 0.0
P_wrong, Q_wrong = fit_mf_sgd(R5, rank=5, steps=250, eta=0.02, reg=0.01, seed=5, missing_as_zero=True)
P_fixed, Q_fixed = fit_mf_sgd(R5, rank=2, steps=250, eta=0.03, reg=0.08, seed=5, missing_as_zero=False)
truth = [true_rating for user, item, true_rating in d5["holdout"]]
wrong = [np.clip(mf_predict(P_wrong, Q_wrong, user, item), 1.0, 5.0) for user, item, true_rating in d5["holdout"]]
fixed = [np.clip(mf_predict(P_fixed, Q_fixed, user, item), 1.0, 5.0) for user, item, true_rating in d5["holdout"]]

print("wrong missing-as-zero RMSE", round(rmse(truth, wrong), 3))
print("fixed observed-only RMSE", round(rmse(truth, fixed), 3))


## Evaluate it + Practice

Evaluation checklist:
- Track `RMSE` on D1--D5 and compare with a no-skill baseline such as popularity or original order.
- Run a sanity check where the strongest observed signal is swapped and the top item changes.
- Ablate the key idea, such as masking, latent factors, calibration, or query-local losses, and confirm the metric drops.
- Watch failure signals: identical recommendations for everyone, head-item dominance, unstable tiny-overlap scores, and poor cold-start behavior.

Practice:
1. Change one D3 tie and predict which slate position moves before running the cell.


2. Try rank 1, 2, and 4 on D4 and compare validation RMSE.
3. Increase regularization until latent factors shrink visibly.